# Step 3: ML 파이프라인 추가

<div class="alert alert-warning"> 이 노트북은 <code>SageMaker Distribution Image 3.7.0</code> 을 사용하는 SageMaker Studio JupyterLab 인스턴스와 SageMaker Python SDK 버전 <code>2.255.0</code></div>

이 단계에서는 [Amazon SageMaker Pipelines](https://aws.amazon.com/sagemaker/pipelines/) 와 [Amazon SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). 를 사용하여 엔드투엔드 ML 워크플로를 자동화합니다. [Amazon SageMaker Feature Store](https://aws.amazon.com/sagemaker/feature-store/).

||||
|---|---|---|
|1. |노트북에서 실험 ||
|2. |SageMaker AI 처리 작업 및 SageMaker SDK로 확장 ||
|3. |ML 파이프라인, 모델 레지스트리, 피처 스토어로 운영화 |**<<<< 현재 위치**|
|4. |모델 빌드 CI/CD 파이프라인 추가 ||
|5. |모델 배포 파이프라인 추가 ||
|6. |모델 및 데이터 모니터링 추가 ||

<div class="alert alert-info"> 이 노트북에서는 <code>Python 3</code> 커널을 사용해야 합니다.</div>

In [ ]:
# Feature Processor 모듈을 가져오기 위해 추가 종속성과 함께 SageMaker Python SDK를 다시 설치해야 합니다
%pip install 'sagemaker[feature-processor]==2.255.0' --force-reinstall

In [ ]:
%pip install --force-reinstall --no-cache s3fs boto3

In [ ]:
# %pip install --force-reinstall --no-cache boto3

In [ ]:
# 커널 재시작
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import p와as as pd
import json
import boto3
import pathlib
import io
import sagemaker
import mlflow
from time import gmtime, strftime, sleep
from sagemaker.deserializers import CSVDeserializer
from sagemaker.serializers import CSVSerializer
from importlib.metadata import version, PackageNotFoundError

from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.xgboost.estimator import XGBoost
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import (
    ProcessingInput, 
    ProcessingOutput, 
    ScriptProcessor
)
from sagemaker.inputs import TrainingInput

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import (
    ProcessingStep, 
    TrainingStep, 
    CreateModelStep,
    CacheConfig
)
from sagemaker.workflow.check_job_config import CheckJobConfig
from sagemaker.workflow.parameters import (
    ParameterInteger, 
    ParameterFloat, 
    ParameterString, 
    ParameterBoolean
)
from sagemaker.workflow.quality_check_step import (
    DataQualityCheckConfig,
    ModelQualityCheckConfig,
    QualityCheckStep,
)
from sagemaker.workflow.clarify_check_step import (
    ModelBiasCheckConfig, 
    ClarifyCheckStep, 
    ModelExplainabilityCheckConfig
)
from sagemaker import Model
from sagemaker.inputs import CreateModelInput
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.conditions import (
    ConditionGreaterThan,
    ConditionGreaterThanOrEqualTo
)
from sagemaker.workflow.parallelism_config import ParallelismConfiguration
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import (
    Join,
    JsonGet
)
from sagemaker.workflow.lambda_step import (
    LambdaStep,
    LambdaOutput,
    LambdaOutputTypeEnum,
)
from sagemaker.lambda_helper import Lambda

from sagemaker.model_metrics import (
    MetricsSource, 
    ModelMetrics, 
    FileSource
)
from sagemaker.drift_check_baselines import DriftCheckBaselines
from sagemaker.workflow.pipeline_definition_config import PipelineDefinitionConfig 
from sagemaker.image_uris import retrieve
from sagemaker.workflow.function_step import step
from sagemaker.workflow.step_outputs import get_step
from sagemaker.model_monitor import DatasetFormat, model_monitoring
from IPython.display import HTML

(sagemaker.__version__, boto3.__version__, mlflow.__version__)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here 노트북   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

## 상수 설정

In [ ]:
# 파이프라인 객체, 실험 및 모델 이름 설정
project = "from-idea-to-prod"

current_timestamp = strftime('%d-%H-%M-%S', gmtime())

registered_model_name = f"{project}-pipeline-model-{current_timestamp}"
experiment_name = f"{project}-pipeline-{current_timestamp}"
pipeline_name = f"{project}-pipeline-{current_timestamp}"
pipeline_model_name = f"{project}-model-xgb"
model_package_group_name = registered_model_name
endpoint_config_name = f"{project}-endpoint-config"
endpoint_name = f"{project}-endpoint"
model_approval_status = "PendingManualApproval"

In [ ]:
# 인스턴스 유형 및 개수 설정
process_instance_type = "ml.m5.large"
train_instance_type = "ml.m5.large"

In [ ]:
# 파이프라인에서 생성된 다양한 데이터셋을 위한 S3 URL 설정
output_s3_prefix = f"s3://{bucket_name}/{bucket_prefix}"
output_s3_url = f"{output_s3_prefix}/output"

train_s3_url = f"{output_s3_prefix}/train"
validation_s3_url = f"{output_s3_prefix}/validation"
test_s3_url = f"{output_s3_prefix}/test"
evaluation_s3_url = f"{output_s3_prefix}/evaluation"

baseline_s3_url = f"{output_s3_prefix}/baseline"
baseline_results_s3_url = f"{baseline_s3_url}/results"

prediction_baseline_s3_url = f"{output_s3_prefix}/prediction_baseline"
prediction_baseline_results_s3_url=f"{prediction_baseline_s3_url}/results"

In [ ]:
XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(
            "xgboost", 
            region=boto3.Session().region_name,
            version="1.7-1"
)

In [ ]:
%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url
%store pipeline_name
%store model_package_group_name
%store evaluation_s3_url
%store prediction_baseline_s3_url
%store output_s3_url

In [ ]:
print(f"Train S3 url: {train_s3_url}")
print(f"Validation S3 url: {validation_s3_url}")
print(f"Test S3 url: {test_s3_url}")
print(f"Data baseline S3 url: {baseline_s3_url}")
print(f"Evaluation metrics S3 url: {evaluation_s3_url}")
print(f"Model prediction baseline S3 url: {prediction_baseline_s3_url}")

## 헬퍼 함수 정의
코드 가독성을 높이기 위한 간단한 함수를 정의합니다.

In [ ]:
def get_xgb_estimator(
    session,
    instance_type,
    output_s3_url,
    base_job_name,
):
    # Instantiate an XGBoost estimator object
    estimator = sagemaker.estimator.Estimator(
        image_uri=XGBOOST_IMAGE_URI,
        role=sagemaker.get_execution_role(), 
        instance_type=instance_type,
        instance_count=1,
        output_path=output_s3_url,
        sagemaker_session=session,
        base_job_name=base_job_name,
    )
    
    # Define algorithm hyperparameters
    estimator.set_hyperparameters(
        num_round=100, # the number of rounds to run the training
        max_depth=3, # maximum depth of a tree
        eta=0.5, # step size shrinkage used in updates to prevent overfitting
        alpha=2.5, # L1 regularization term on weights
        objective="binary:logistic",
        eval_metric="auc", # evaluation metrics for validation data
        subsample=0.8, # subsample ratio of the training instance
        colsample_bytree=0.8, # subsample ratio of columns when constructing each tree
        min_child_weight=3, # minimum sum of instance weight (hessian) needed in a child
        early_stopping_rounds=10, # the model trains until the validation score stops improving
        verbosity=1, # verbosity of printing messages
    )

    return estimator

## MLflow 구성

In [ ]:
# Check that the MLflow server is 폴더의 status 'Created' or 'Started'
sm = boto3.client("sagemaker")

while sm.describe_mlflow_app(Arn=mlflow_arn)['Status'] not in ['Created', 'Updated']:
    print(f"The MLflow server {mlflow_name} is not 폴더의 status 'Created' or 'Started'")
    sleep(30)
else:
    print(f"Using server {mlflow_name}")

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
experiment = mlflow.set_experiment(experiment_name=experiment_name)

## AWS 인프라 기본값 구성
YAML 구성 파일을 사용하여 작업 파라미터와 같이 SageMaker API에 자동으로 전달되는 기본값을 정의할 수 있습니다. VPC ID, 보안 그룹, KMS 키 등과 같은 인프라 설정을 위한 정적 파라미터를 제공하거나 원격 함수로 작업할 때 특히 유용합니다.

Refer to [Configuring 와 using defaults 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-와-using-defaults-처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다.-the-sagemaker-python-sdk) documentation for examples 와 more details.

Your SageMaker pipeline will use these `config.yaml` files.

In [ ]:
# 구성 파일의 기본 위치 출력
import os
from platformdirs import site_config_dir, user_config_dir

#Prints the location of the admin config file
print(os.path.join(site_config_dir("sagemaker"), "config.yaml"))

#Prints the location of the user config file
print(os.path.join(user_config_dir("sagemaker"), "config.yaml"))

The next cell creates a configuration file 와 sets default values for remote functions. 이러한 값은 `@step` decorator 와 you don't need to specify them explicitly. Refer to [Configure your pipeline](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator-cfg-pipeline.html) 폴더의 Developer Guide.

In [ ]:
%%writefile config.yaml

SchemaVersion: '1.0'
SageMaker:
    PythonSDK:
        Modules:
            RemoteFunction:
                InstanceType: ml.m5.xlarge
                Dependencies: ./requirements.txt
                IncludeLocalWorkDir: true
                CustomFileFilter:
                    IgnoreNamePatterns: # files or directories to ignore
                        - "*.ipynb" # all 노트북 files
                        - "*.md" # all markdown files
                        - "__pycache__"

In [ ]:
# 구성 파일을 사용자 구성 파일 위치로 복사
%mkdir -p {user_config_dir("sagemaker")}
%cp config.yaml {os.path.join(user_config_dir("sagemaker"), "config.yaml")}

In [ ]:
# or config.yaml을 사용자 구성 디렉토리로 복사하는 대신 SageMaker가 구성 파일을 가리키도록 할 수 있습니다
# os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = os.getcwd()

## 환경 준비

In [ ]:
# 코드의 로컬 테스트를 위해 xgboost 설치
%pip install -q xgboost

Get the version of the installed packages 와 create a `requirements.txt` 파일을 생성하여 환경을 복제합니다. SageMaker 파이프라인은 이를 사용하여 작업 컨테이너의 환경을 설정합니다.

In [ ]:
if os.path.exists('requirements.txt'):
    os.remove('requirements.txt')
    print("Existing requirements.txt file deleted.")

In [ ]:
packages = ['xgboost', 'mlflow', 'sagemaker-mlflow','sagemaker']
requirements = [f'{p}=={version(p)}' for p in packages]

requirements.append('protobuf==3.20.1')
requirements.append('s3fs==0.4.2') #FIXME: find a more up to date version that works

if requirements:
    처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. open('requirements.txt', 'w') as f:
        f.write('\n'.join(requirements))
    print("\nNew requirements.txt file created 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the following content:")
    print('\n'.join(requirements))
else:
    print("\nNo requirements.txt file created as no packages were found")

In [ ]:
# %%writefile requirements.txt
# scikit-learn
# p와as>=2.0.0
# s3fs==0.4.2
# sagemaker>=2.237
# xgboost
# mlflow==2.16.2
# sagemaker-mlflow==0.1.0

## SageMaker 파이프라인

### 파이프라인 파라미터 설정
SageMaker Pipelines는 [parameterization](https://docs.aws.amazon.com/sagemaker/latest/dg/build-와-manage-parameters.html), 를 지원하며, 이를 통해 파이프라인 코드를 변경하지 않고 런타임에 입력 파라미터를 지정할 수 있습니다. 다음 모듈에서 사용 가능한 파라미터 클래스를 사용할 수 있습니다: [`sagemaker.workflow.parameters`](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#parameters) 모듈.
파라미터에는 기본값이 있으며, 파이프라인 실행을 시작할 때 파라미터 값을 지정하여 재정의할 수 있습니다.

In [ ]:
# 처리 인스턴스 유형 설정
process_instance_type_param = ParameterString(
    name="ProcessingInstanceType",
    default_value=process_instance_type,
)

# 훈련 인스턴스 유형 설정
train_instance_type_param = ParameterString(
    name="TrainingInstanceType",
    default_value=train_instance_type,
)

# 모델 레지스트리의 모델 승인 상태 설정
model_approval_status_param = ParameterString(
    name="ModelApprovalStatus",
    default_value=model_approval_status
)

# Minimal threshold for model performance on the test dataset
test_score_threshold_param = ParameterFloat(
    name="TestScoreThreshold",
    default_value=0.75
)

# 입력 데이터셋을 위한 S3 URL 매개변수화
input_s3_url_param = ParameterString(
    name="InputDataUrl",
    default_value=input_s3_url,
)

# 모델 패키지 그룹 이름
model_package_group_name_param = ParameterString(
    name="ModelPackageGroupName",
    default_value=model_package_group_name,
)

# MLflow 추적 서버 ARN
tracking_server_arn_param = ParameterString(
    name="TrackingServerARN",
    default_value=mlflow_arn,
)

### 파이프라인 단계 구현 및 테스트
다음과 같은 파이프라인을 생성합니다:
| Step | Description |
|---|---|
| **Data processing** | runs a SageMaker processing job for feature engineering 와 dataset split|
| **Training** | XGBoost 알고리즘을 사용하여 SageMaker 훈련 작업 실행 |
| **Evaluation** | 훈련된 모델의 성능 평가 |
| **Conditional step** | 모델의 성능이 지정된 임계값을 충족하는지 확인 |
| **Register model** | SageMaker 모델 레지스트리에 모델 버전 등록 |

파이프라인을 코드로 구현하기 위해 SageMaker의 두 가지 유용한 기능을 사용합니다: Subclass compatibility 와 @step decorator.

#### 서브클래스 호환성
💡 다음을 위해 서브클래스 호환성을 사용할 수 있습니다: [workflow pipeline job steps](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#steps) to build job abstractions 와 use exactly the same code to configure the pipeline as the code for running processing, training, transform, 와 tuning jobs from the previous step 노트북s. 다음을 사용해야 합니다: [PipelineSession](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.pipeline_context.PipelineSession) 대신 `sagemaker_session` 와 같은 실행 호출을 캡처하려면 `processor.run()` or `estimator.fit()` but not run until the pipeline is created 와 executed.

#### @step 데코레이터
💡 You can lift-와-shift your existing Python code to SageMaker pipelines. You can also use Python functions to implement an ML workflow using SageMaker Python SDK 와 test all code locally 폴더의 노트북. 파이프라인을 생성하려면 SageMaker Python SDK [`@step`](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#step-decorator) 데코레이터를 사용하여 Python 함수를 파이프라인 단계로 변환할 수 있습니다. Refer to the SageMaker [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator-create-pipeline.html) for more details 와 examples.

The following code uses Python functions to implement workflow steps, test them locally, 와 then apply `@step` 데코레이터를 적용하여 함수를 파이프라인 단계로 재사용합니다.

#### @step 데코레이터 사용 시 제한사항
다음의 특정 제한사항에 유의하세요: [limitations](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator-limit.html) 를 사용할 때 `@step` 데코레이터를 사용할 때의 제한사항.

다음 코드 셀에서 단계 개발 및 테스트를 시작합니다.

#### 전처리 단계
2단계 [노트북](./02-sagemaker-containers.ipynb) 와 create a `preprocess` 폴더에 `./pipeline_steps` 파일에 있습니다. 이 함수를 사용하여 파이프라인 처리 단계를 생성합니다.

In [ ]:
# Python 함수 코드는 로컬 파일에 있습니다
from pipeline_steps.preprocess import preprocess

In [ ]:
# 함수 코드 보기
# !pygmentize pipeline_steps/preprocess.py

In [ ]:
# S3 URL에 데이터셋이 있는지 확인
!aws s3 ls {input_s3_url}

In [ ]:
# 파이프라인을 구성하기 전에 Python 코드를 로컬에서 실행하고 정확성을 확인할 수 있습니다
r_preprocess = preprocess(
    input_data_s3_path=input_s3_url,
    output_s3_prefix=output_s3_prefix,
    tracking_server_arn=mlflow_arn,
    experiment_name=f"local-test-{current_timestamp}"
)
r_preprocess

In [ ]:
# 함수가 출력을 생성했는지 확인
!aws s3 ls {output_s3_prefix}/test/

#### 훈련 단계
먼저, SageMaker 내장 알고리즘 훈련 작업으로 모델 훈련을 원격으로 실행합니다. 

둘째, 훈련된 모델을 사용하여 평가 스크립트를 로컬 Python 함수로 테스트합니다.

셋째, 이 Python 함수를 사용하여 파이프라인에서 평가 단계를 구성합니다.

In [ ]:
# 추정기에서 sagemaker.Session()을 사용하여 훈련 작업을 즉시 실행
estimator = get_xgb_estimator(
    session=sagemaker.Session(),
    instance_type=train_instance_type,
    output_s3_url=output_s3_url,
    base_job_name=f"{project}-train",
)

In [ ]:
# 전처리 함수의 출력을 사용하여 훈련 입력 설정
training_inputs = {
    "train": TrainingInput(
        s3_data=r_preprocess['train_data'],
        content_type="text/csv",
    ),
    "validation": TrainingInput(
        s3_data=r_preprocess['validation_data'],
        content_type="text/csv",
    ),
}

다음 코드 셀은 추정기를 피팅합니다. 훈련 작업이 완료될 때까지 3분 미만 기다립니다. 코드가 훈련된 모델을 MLflow 실험 실행에 로깅한다는 점에 유의하세요.

In [ ]:
from pipeline_steps.evaluate import load_model

# 훈련 작업 실행
mlflow.set_experiment(r_preprocess['experiment_name'])
처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. mlflow.start_run(
    run_name=f"training-{strftime('%d-%H-%M-%S', gmtime())}",
    description="training 폴더의 노트북 03 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. a training job") as run:
    mlflow.log_params(estimator.hyperparameters())
    
    estimator.fit(training_inputs)

    mlflow.log_param("training job name", estimator.latest_training_job.name)
    mlflow.log_metrics({i['metric_name'].replace(':', '_'):i['value'] for i in estimator.training_job_analytics.dataframe().iloc})
    mlflow.xgboost.log_model(load_model(estimator.model_data), artifact_path="xgboost")

#### 평가 단계
모델 성능이 지정된 임계값을 충족하는지 확인하기 위한 로컬 Python 함수로 모델 평가 스크립트를 생성합니다. Python 코드는 `evaluate.py` 폴더의 `./pipeline_steps` 파일에 있습니다.

In [ ]:
from pipeline_steps.evaluate import evaluate

In [ ]:
# !pygmentize  pipeline_steps/evaluation.py

In [ ]:
# 방금 실행한 훈련 작업이 모델 파일을 생성했는지 확인
!aws s3 ls {estimator.model_data}

Now load the trained model 폴더의 evaluation script 와 verify that the script executes correctly. 다음의 출력 파라미터 사용에 주목하세요: `preprocess` function 와 the model artifact from the training job.

In [ ]:
# 평가 코드를 로컬에서 실행
r_eval = evaluate(
    test_x_data_s3_path=r_preprocess['test_x_data'],
    test_y_data_s3_path=r_preprocess['test_y_data'],
    model_s3_path=estimator.model_data,
    output_s3_prefix=output_s3_prefix,
    tracking_server_arn=mlflow_arn,
    experiment_name=r_preprocess['experiment_name'],
)
r_eval

In [ ]:
# 평가 함수가 출력을 생성했는지 확인
!aws s3 ls {output_s3_prefix}/prediction_baseline/

#### 모델 등록 단계
The register step creates a SageMaker model 와 registers a new version of a model 폴더의 SageMaker Model Registry 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다.in a [model package group](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-model-group.html). 
이 단계도 로컬 Python 함수로 구현합니다. The code is provided 폴더의 file `register.py` 폴더의 `./pipeline_steps` 파일에 있습니다.

In [ ]:
from pipeline_steps.register import register

In [ ]:
# !pygmentize  pipeline_steps/register.py

In [ ]:
# 모델 등록 코드를 로컬에서 실행
r_register = register(
    training_job_name=estimator.latest_training_job.name,
    model_package_group_name=model_package_group_name,
    model_approval_status=model_approval_status,
    evaluation_result=r_eval['evaluation_result'],
    output_s3_prefix=output_s3_url,
    tracking_server_arn=mlflow_arn,
    experiment_name=r_preprocess['experiment_name'],
)
r_register

In [ ]:
# 모델 패키지 그룹에 새 모델 버전이 등록되었는지 확인
boto3.client('sagemaker').describe_model_package(ModelPackageName=r_register['model_package_arn'])

### 파이프라인 구성
로컬 테스트 후 동일한 Python 코드를 변경 없이 사용하여 파이프라인을 구성할 수 있습니다.

The next cell creates a pipeline 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. previously developed 와 tested steps. 참고로 `@step`데코레이터가 적용된 함수 (preprocess, evaluate, register) 와 traditional pipeline steps (train) as SageMaker jobs 폴더의 same pipeline 와 pass data between them.

SageMaker가 파이프라인 단계 간의 데이터 종속성을 기반으로 처리 흐름을 자동으로 파생하므로 단계의 순서를 수동으로 정의할 필요가 없습니다. You also don't need to manage transfer of artifacts 와 datasets from one pipeline's step to another, because SageMaker automatically takes care of the data flow.

다음의 사용에 주목하세요: [`PipelineSession`](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.pipeline_context.PipelineSession) instead of Session 폴더의 estimator object for the training step. 파이프라인을 구성할 때 PipelineSession 객체를 `Estimator` or `Processor` to start the job at pipeline execution time 와 not immedeately as 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. SageMaker `Session`.

In [ ]:
# 데이터 전처리 단계
step_preprocess = step(
    preprocess, 
    instance_type=process_instance_type_param,
    name=f"{project}-preprocess",
    keep_alive_period_in_seconds=3600,
)(
    input_data_s3_path=input_s3_url_param,
    output_s3_prefix=output_s3_prefix,
    tracking_server_arn=tracking_server_arn_param,
    experiment_name=experiment_name,
    pipeline_run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
)

cache_config = CacheConfig(enable_caching=True)
cache_config.expire_after = "p30d"

# 훈련 단계
step_train = TrainingStep(
    name=f"{project}-train",
    step_args=get_xgb_estimator(
        session=PipelineSession(),
        instance_type=train_instance_type_param,
        output_s3_url=output_s3_url,
        base_job_name=f"{project}-train",
    ).fit(
        {
            "train": TrainingInput(
                step_preprocess['train_data'],
                content_type="text/csv",
            ),
            "validation": TrainingInput(
                step_preprocess['validation_data'],
                content_type="text/csv",
            ),
        }
    ),
    cache_config=cache_config,
)    

# 평가 단계
step_evaluate = step(
    evaluate,
    instance_type=process_instance_type_param,
    name=f"{project}-evaluate",
    keep_alive_period_in_seconds=3600,
)(
    test_x_data_s3_path=step_preprocess['test_x_data'],
    test_y_data_s3_path=step_preprocess['test_y_data'],
    model_s3_path=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    output_s3_prefix=output_s3_prefix,
    tracking_server_arn=tracking_server_arn_param,
    experiment_name=step_preprocess['experiment_name'],
    pipeline_run_id=step_preprocess['pipeline_run_id'],
)

# 모델 등록 단계
step_register = step(
        register,
        name=f"{project}-register",
        keep_alive_period_in_seconds=3600,
    )(
        training_job_name=step_train.properties.TrainingJobName,
        model_package_group_name=model_package_group_name_param,
        model_approval_status=model_approval_status_param,
        evaluation_result=step_evaluate['evaluation_result'],
        output_s3_prefix=output_s3_url,
        tracking_server_arn=tracking_server_arn_param,
        experiment_name=step_preprocess['experiment_name'],
        pipeline_run_id=step_preprocess['pipeline_run_id'],
    )

# 파이프라인 실행 실패 단계
step_fail = FailStep(
    name=f"{project}-fail",
    error_message=Join(on=" ", values=["Execution failed due to AUC Score < ", test_score_threshold_param]),
)

# 조건 단계에서 확인할 조건
condition_gte = ConditionGreaterThanOrEqualTo(
        left=step_evaluate['evaluation_result']['classification_metrics']['auc_score']['value'],  
        right=test_score_threshold_param,
)

# 조건부 등록 단계
step_conditional_register = ConditionStep(
    name=f"{project}-check-metrics",
    conditions=[condition_gte],
    if_steps=[step_register],
    else_steps=[step_fail],
)

# 파이프라인 객체 생성
pipeline = Pipeline(
    name=f"{pipeline_name}",
    parameters=[
        input_s3_url_param,
        process_instance_type_param,
        train_instance_type_param,
        model_approval_status_param,
        test_score_threshold_param,
        model_package_group_name_param,
        tracking_server_arn_param,
    ],
    steps=[step_conditional_register],
    pipeline_definition_config=PipelineDefinitionConfig(use_custom_job_prefix=True)
)

Note, we added two more steps to the pipeline: Fail step 와 Condition Step.

#### Fail step
Pipelines [FailStep](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.fail_step.FailStep) stops the pipeline execution if the model performance metric doesn't meet the specified threshold. 

#### Condition step
The condition step checks the model performance score calculated 폴더의 evaluation step 와 conditionally creates a model 와 registers it 폴더의 model registry, or stops 와 fails the pipeline execution.

#### 파이프라인 생성/업데이트
이제 파이프라인을 생성합니다. 동일한 이름의 파이프라인이 이미 존재하면 SageMaker가 업데이트합니다. 

다음에 마지막 단계만 전달하면 됩니다: `Pipeline` constructor. SDK는 단계 간의 데이터 종속성을 기반으로 파이프라인 DAG를 자동으로 구축합니다. Refer to the [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator-create-pipeline.html#pipelines-step-define-delayed) for more details.

In [ ]:
# Upsert 작업은 파이프라인 런타임 중에 액세스할 수 있도록 함수 코드, 인수 및 기타 아티팩트를 S3로 직렬화합니다
pipeline.upsert(role_arn=sm_role)

To see the created pipeline 폴더의 Studio UI, click on the link constructed by the code cell below:

In [ ]:
# 파이프라인 링크 표시
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/graph">the pipeline</a> 폴더의 Studio UI</b>'.format(
            domain_id, region, pipeline_name))
)

Based on the data dependencies between the pipeline's steps, SageMaker builds the following DAG 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the data flow in your pipeline:
![](img/pipeline-graph.png)

### 파이프라인 실행
첫 번째 파이프라인 실행은 약 17-20분이 소요됩니다. 다음의 사용에 주목하세요: `keep_alive_period_in_seconds` parameter 폴더의 step definition for the warm pool reuse 와 `CacheConfig` 폴더의 Training step for the caching of step results.
A subsequent pipeline execution takes about 7 minutes due to usage of caching 와 a warm pool.

In [ ]:
pipeline_execution = pipeline.start()
pipeline_execution.describe()

In [ ]:
# 이 실행이 완료될 때까지 노트북에서 기다리려면 주석 해제
# pipeline_execution.wait() 
pipeline_execution.list_steps()

You can see the pipeline execution 폴더의 Studio UI by clicking on the link constructed by the following code cell:

In [ ]:
# 파이프라인 실행 링크 표시
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/executions/{}/graph">the pipeline execution</a> 폴더의 Studio UI</b>'.format(
            domain_id, region, pipeline_name, pipeline_execution.describe()['PipelineExecutionArn'].split('/')[-1]))
)

To manage pipelines 폴더의 Studio UI select **Pipelines** 폴더의 navigation menu on the left 와 then the specific pipeline to see pipeline's executions 와 all details:

![](img/pipelines-pane.png)

For each execution you can open an execution graph 와 see details of each pipeline 단계:

![](img/pipeline-execution-graph.png)

You can track pipeline executions, artifacts, datasets, 와 models 폴더의 MLflow UX:

![](img/mlflow-pipeline-executions.png)

### 파이프라인 정의 이해

In [ ]:
pipeline_definition = json.loads(pipeline.describe()['PipelineDefinition'])
pipeline_definition

Look at 와 underst와 the pipeline definition JSON. For example, you can see, how the pipeline paramemters are defined 와 how are they used.

정의:
```json
'Parameters': [{'Name': 'ProcessingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.large'},
               ...
               ]
```

파라미터 대체:
```json
'Arguments': {'ProcessingResources': {'ClusterConfig': {'InstanceType':{'Get':'Parameters.ProcessingInstanceType'}
```

이제 다음의 정의를 찾으세요: `preprocess` 단계:

```json
{'Name': 'from-idea-to-prod-preprocess',
   'Type': 'Training',
   'Arguments': {'TrainingJobName': 'preprocess',
    'RoleArn': 'arn:aws:iam::906545278380:role/service-role/AmazonSageMaker-ExecutionRole-20240214T222844',
    'StoppingCondition': {'MaxRuntimeInSeconds': 86400},
    'RetryStrategy': {'MaximumRetryAttempts': 1},
    'InputDataConfig': [{'ChannelName': 'sagemaker_remote_function_bootstrap',
      'DataSource': {'S3DataSource': {'S3Uri': 's3://sagemaker-us-east-1-906545278380/from-idea-to-prod-pipeline-03-09-21-09/sagemaker_remote_function_bootstrap',
        'S3DataType': 'S3Prefix'}}},
```

You see that SageMaker by default runs your Python script in a container 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the same image as the kernel image used for this JupyterLab 노트북:

```json
'AlgorithmSpecification': {'TrainingImage': '885854791233.dkr.ecr.us-east-1.amazonaws.com/sagemaker-distribution-prod@sha256:7c07530831d3d25b27a77b6a77f9801eec01b7b80c69ca1aa2c9eae3df00887d',
```

If you want to create pipelines by h와, you can work 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. JSON 와 follow the [SageMaker Pipeline Definition JSON Schema](https://aws-sagemaker-mlops.github.io/sagemaker-model-building-pipeline-definition-JSON-schema/index.html).

## 파이프라인에 배치 변환 및 품질 검사 추가
필요한 모든 작업을 자동화하기 위해 모델 빌드 파이프라인에 추가 단계를 통합할 수 있습니다. 이 섹션에서는 다음 단계를 추가합니다:
- Quality checks for both data 와 the model 와 baseline calculation using [`QualityCheckStep`](https://docs.aws.amazon.com/sagemaker/latest/dg/build-와-manage-steps.html#step-type-quality-check)
- Batch transform using [`TransformStep`](https://docs.aws.amazon.com/sagemaker/latest/dg/build-와-manage-steps.html#step-type-transform)

For a more detailed example 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. model quality checks refer to an example 노트북 [SageMaker Pipelines integration 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. Model Monitor 와 Clarify](https://github.com/aws/amazon-sagemaker-examples/blob/main/sagemaker-pipelines/tabular/model-monitor-clarify-pipelines/sagemaker-pipeline-model-monitor-clarify-steps.ipynb).


To underst와 the data 와 model quality life cycle refer to the Developer Guide [Baseline calculation, drift detection 와 lifecycle 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. ClarifyCheck 와 QualityCheck steps in Amazon SageMaker Model Building Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-quality-clarify-baseline-lifecycle.html).

In [ ]:
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import CreateModelInput

### Quality checks
Start 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. definition of data 와 model quality check steps for the pipeline. The data 와 model quality check steps use data from `preprocess` 와 `evaluate` 파이프라인의 단계.

In [ ]:
# 데이터 품질 검사 제어 파라미터
skip_check_data_quality_param = ParameterBoolean(name="SkipDataQualityCheck", default_value=True)
register_new_baseline_data_quality_param = ParameterBoolean(
    name="RegisterNewDataQualityBaseline", default_value=True
)

# 모델 품질 검사 제어 파라미터
skip_check_model_quality_param = ParameterBoolean(name="SkipModelQualityCheck", default_value=True)
register_new_baseline_model_quality_param = ParameterBoolean(
    name="RegisterNewModelQualityBaseline", default_value=True
)

# 데이터 및 모델 품질 검사 단계를 위한 작업 구성
check_job_config = CheckJobConfig(
    role=sm_role,
    instance_count=1,
    instance_type=process_instance_type_param,
)

# 데이터 품질 검사 단계 구성
data_quality_check_config = DataQualityCheckConfig(
    baseline_dataset=step_preprocess['baseline_data'],
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=baseline_results_s3_url,
)

# 모델 품질 검사 단계 구성
model_quality_check_config = ModelQualityCheckConfig(
    baseline_dataset=step_evaluate['prediction_baseline_data'],
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=prediction_baseline_results_s3_url,
    problem_type="BinaryClassification",
    inference_attribute= "prediction", # The column 폴더의 dataset that contains predictions
    probability_attribute= "probability", # The column 폴더의 dataset that contains probabilities
    ground_truth_attribute= "label", # The column 폴더의 dataset that contains ground truth labels
)

cache_config = CacheConfig(enable_caching=True)
cache_config.expire_after = "p30d"

# 데이터 품질 검사 단계
step_data_quality_check = QualityCheckStep(
    name=f"{project}-data-quality",
    quality_check_config=data_quality_check_config,
    check_job_config=check_job_config,
    skip_check=skip_check_data_quality_param,
    register_new_baseline=register_new_baseline_data_quality_param,
    model_package_group_name=model_package_group_name_param,
    cache_config=cache_config,
)

# 모델 품질 검사 단계
step_model_quality_check = QualityCheckStep(
    name=f"{project}-model-quality",
    quality_check_config=model_quality_check_config,
    check_job_config=check_job_config,
    skip_check=skip_check_model_quality_param,
    register_new_baseline=register_new_baseline_model_quality_param,
    model_package_group_name=model_package_group_name_param,
    cache_config=cache_config,
)

### 배치 변환
추가 a transform step to the pipeline.

In [ ]:
session = PipelineSession()

# create model step
step_create_model = ModelStep(
    name=f"{project}-model",
    step_args=Model(
        image_uri=XGBOOST_IMAGE_URI,        
        model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
        name=f"from-idea-to-prod-xgboost-model",
        sagemaker_session=session,
        role=sm_role,
    ).create(instance_type="ml.m5.large"),
)

# 변환 단계 생성
step_transform = TransformStep(
    name=f"{project}-transform", 
    step_args=Transformer(
        model_name=step_create_model.properties.ModelName,
        instance_type=train_instance_type_param,
        instance_count=1,
        accept="text/csv",
        assemble_처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다.="Line",
        output_path=f"{output_s3_prefix}/transform",
        sagemaker_session=session,
        base_transform_job_name=f"{project}-transform",
    ).transform(    
        data=step_preprocess["test_x_data"],
        content_type="text/csv",
        split_type="Line", 
    ),
    cache_config=cache_config,
)

You need to include the generated model 와 data quality baselines into the model register step.

In [ ]:
# 계산된 모델 및 데이터 품질 베이스라인을 사용하도록 모델 등록 단계 재정의
step_register = step(
        register,
        name=f"{project}-register",
        keep_alive_period_in_seconds=3600,
    )(
        training_job_name=step_train.properties.TrainingJobName,
        model_package_group_name=model_package_group_name_param,
        model_approval_status=model_approval_status_param,
        evaluation_result=step_evaluate['evaluation_result'],
        output_s3_prefix=output_s3_url,
        tracking_server_arn=tracking_server_arn_param,
        model_statistics_s3_path=step_model_quality_check.properties.CalculatedBaselineStatistics,
        model_constraints_s3_path=step_model_quality_check.properties.CalculatedBaselineConstraints,
        model_data_statistics_s3_path=step_data_quality_check.properties.CalculatedBaselineStatistics,
        model_data_constraints_s3_path=step_data_quality_check.properties.CalculatedBaselineConstraints,
        experiment_name=step_preprocess['experiment_name'],
        pipeline_run_id=step_preprocess['pipeline_run_id'],
    )

# 새 step_register로 조건부 등록 단계 재정의
step_conditional_register = ConditionStep(
    name=f"{project}-check-metrics",
    conditions=[condition_gte],
    if_steps=[step_register, step_transform],
    else_steps=[step_fail],
)

In [ ]:
# 파이프라인 객체 생성
pipeline = Pipeline(
    name=f"{pipeline_name}",
    parameters=[
        input_s3_url_param,
        process_instance_type_param,
        train_instance_type_param,
        model_approval_status_param,
        test_score_threshold_param,
        model_package_group_name_param,
        tracking_server_arn_param,
        skip_check_data_quality_param,
        skip_check_model_quality_param,
        register_new_baseline_data_quality_param,
        register_new_baseline_model_quality_param,
    ],
    steps=[step_conditional_register],
    pipeline_definition_config=PipelineDefinitionConfig(use_custom_job_prefix=True)
)

In [ ]:
# 파이프라인 업데이트
pipeline.upsert(role_arn=sm_role, parallelism_config=ParallelismConfiguration(5).to_request())

In [ ]:
# 파이프라인 링크 표시
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/graph">the pipeline</a> 폴더의 Studio UI</b>'.format(
            domain_id, region, pipeline_name))
)

The new pipeline contains now the steps for data 와 model quality checks 와 the batch transform:
![](img/pipeline-graph-처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다.-transform.png)

### 새 파이프라인 실행
파이프라인이 처음 실행될 때 파라미터 `skip_check_...` 와 `register_new_baseline_...` 는 기본값으로 설정해야 합니다 `(True, True)` so that the quality checks are skipped 와 newly calculated baselines are registered for the model version.

In [ ]:
pipeline_execution = pipeline.start()
pipeline_execution.describe()

In [ ]:
# 이 실행이 완료될 때까지 노트북에서 기다리려면 주석 해제
# pipeline_execution.wait() 
pipeline_execution.list_steps()

If you'd like to execute the pipeline one more time, you can set the `SkipDataQualityCheck` 파라미터를 `True` to run the data quality check by comparing the generated baseline 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the input dataset 와 model output.

In [ ]:
# 한 번 더 실행하려면 주석 해제
# pipeline_execution = pipeline.start(
#     parameters=dict(
#         SkipDataQualityCheck=False,
#     )
# )

### SageMaker Pipelines의 추가 기능
Feel free to explore more useful features of SageMaker Pipelines on your own, such as [selective execution of pipeline steps](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-selective-ex.html), [cross-account support](https://docs.aws.amazon.com/sagemaker/latest/dg/build-와-manage-xaccount.html), [scheduled pipeline runs](https://docs.aws.amazon.com/sagemaker/latest/dg/pipeline-eventbridge.html), or [local mode](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-local-mode.html).

If you need to add more monitoring functionality to the pipeline, you can use the [`MonitorBatchTransformStep`](https://sagemaker.readthedocs.io/en/stable/workflows/pipelines/sagemaker.workflow.pipelines.html#sagemaker.workflow.monitor_batch_transform_step.MonitorBatchTransformStep) to combine a transform step 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. quality checks, bias detection, 와 explainability report.

---

## 옵션: 피처 스토어 추가
In this section 다음을 사용합니다: SageMaker Feature Store to manage features 와 the dataset for model training. 

<div class="alert alert-info"> This section is optional 와 not required for course of the workshop. You can stop here 와 go to the next 노트북.</div>

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.inputs import TableFormatEnum
from sagemaker.feature_store.feature_processor import CSVDataSource, feature_processor, to_pipeline
from sagemaker.remote_function import remote
from sagemaker.workflow.function_step import step
import numpy as np
import p와as as pd
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timezone, date
from time import gmtime, strftime, sleep
import time

In [ ]:
%store -r 

In [ ]:
session = sagemaker.Session()
project = "from-idea-to-prod"
current_timestamp = strftime('%d-%H-%M-%S', gmtime())

In [ ]:
feature_store_bucket_prefix = 'from-idea-to-prod/feature-store'
%store feature_store_bucket_prefix

### 원시 데이터를 훈련 준비가 된 피처로 변환
First transform raw data into features in order to be able to extract the schema from the dataset. You need the data schema for definition of a feature group.

In [ ]:
# 원본 원시 데이터 로드
df_data = pd.read_csv(dataset_file_local_path, sep=";")
pd.set_option("display.max_columns", 500)
df_data

Apply feature engineering to the raw data:

In [ ]:
target_col = "y"

# pdays가 999 값을 가질 때를 포착하는 지표 변수
df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

# 적극적으로 고용되지 않은 개인을 위한 지표
df_data["not_working"] = np.where(
    np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
)

# 불필요한 데이터 제거
df_model_data = df_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)

bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
df_model_data.drop('age', axis=1, inplace=True)
df_model_data.drop('age_range', axis=1, inplace=True)

scaled_features = ['pdays', 'previous', 'campaign']
df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

# "y_no" 및 "y_yes"를 단일 레이블 열로 대체하고 앞으로 가져오기:
df_model_data = pd.concat(
    [
        df_model_data["y_yes"].rename(target_col),
        df_model_data.drop(["y_no", "y_yes"], axis=1),
    ],
    axis=1,
)

In [ ]:
def generate_event_timestamp():
    # 로컬 시간을 나타내는 naive datetime
    naive_dt = datetime.now()
    # 시간대 고려
    aware_dt = naive_dt.astimezone()
    # UTC 시간
    utc_dt = aware_dt.astimezone(timezone.utc)
    # ISO-8601 형식으로 변환
    event_time = utc_dt.isoformat(timespec='milliseconds')
    event_time = event_time.replace('+00:00', 'Z')
    return event_time

<div style="border: 4px solid coral; text-align: center; margin: auto;">
추가 `event_time` 와 `record_id` columns to the dataset as these two fields are required for each feature group:
</div>

In [ ]:
df_model_data['event_time'] = generate_event_timestamp()
df_model_data['record_id'] = [f'R{i}' for i in range(len(df_model_data))]

Feature names cannot contain '.' 와 cannot end on '_'. 또한 열 이름을 변환할 때 열 이름에서 '-'를 제거합니다:

In [ ]:
def convert_col_name(c):
    return c.replace('.', '_').replace('-', '_').rstrip('_')

In [ ]:
df_model_data = df_model_data.rename(columns=convert_col_name)
df_model_data = df_model_data.convert_dtypes(infer_objects=True, convert_boolean=False)
df_model_data['record_id'] = df_model_data['record_id'].astype('string')
df_model_data['event_time'] = df_model_data['event_time'].astype('string')

In [ ]:
df_model_data.dtypes

In [ ]:
df_model_data.columns

In [ ]:
df_model_data.head()

In [ ]:
df_model_data.shape

In [ ]:
df_model_data.to_csv('./data/feature_dataset.csv', index=False)

In [ ]:
record_count = len(df_model_data)

In [ ]:
record_count

### 피처 그룹 생성
Now is everything ready to create a feature group 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the dataset schema.

In [ ]:
dataset_feature_group_name = f'{project}-{current_timestamp}'

In [ ]:
%store dataset_feature_group_name

In [ ]:
dataset_feature_group = FeatureGroup(name=dataset_feature_group_name, sagemaker_session=session)

In [ ]:
# DataFrame을 사용하여 피처 그룹 정의 추출
dataset_feature_group.load_feature_definitions(data_frame=df_model_data)

In [ ]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get('FeatureGroupStatus')
    print(f'Initial status: {status}')
    while status == 'Creating':
        print(f'Waiting for feature group: {feature_group.name} to be created ...')
        time.sleep(5)
        status = feature_group.describe().get('FeatureGroupStatus')
    if status != 'Created':
        raise SystemExit(f'Failed to create feature group {feature_group.name}: {status}')
    print(f'FeatureGroup {feature_group.name} was successfully created.')

In [ ]:
dataset_feature_group.create(
    s3_uri=f's3://{bucket_name}/{feature_store_bucket_prefix}', 
    record_identifier_name='record_id', 
    event_time_feature_name='event_time', 
    role_arn=sm_role, 
    enable_online_store=False,
    table_format=TableFormatEnum.ICEBERG 
)

<div style="border: 4px solid coral; text-align: center; margin: auto;">
Wait until the feature group is created 와 ready for use. It takes less then a minute.
</div>

In [ ]:
wait_for_feature_group_creation_complete(dataset_feature_group)

If you run this workshop in your own account or in any account not provisioned via an AWS-led event, you might have issues 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. creation of a feature group because of permission setup. 다음을 따르세요: [feature group creation troubleshooting guide](https://repost.aws/knowledge-center/sagemaker-featuregroup-troubleshooting). Permission issues are most often connected to missing LakeFormation permissions for the SageMaker execution role. 다음 데이터베이스에 대한 액세스 권한을 부여해야 합니다: `sagemaker_featurestore` SageMaker 실행 역할에.

In [ ]:
dataset_feature_group.describe()

The feature group is ready for use. Now you need to ingest data into it.

### SageMaker 파이프라인을 통해 피처 그룹에 데이터 수집
Same as 폴더의 previous section 다음을 사용합니다: [`@step`](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-step-decorator-create-pipeline.html) decorator to create a feature ingestion pipeline.

Compile all previous feature transformation 와 ingestion code into a remote function. The function code is 폴더의 file `ingest.py` 폴더의 `.\pipeline_steps` 파일에 있습니다. Python SDK를 사용합니다: [`FeatureGroup.ingest()`](https://sagemaker.readthedocs.io/en/stable/api/prep_data/feature_store.html#sagemaker.feature_store.feature_group.FeatureGroup.ingest) method to ingest the content of a p와as DataFrame to a feature group.

In [ ]:
# Python 함수 코드는 로컬 파일에 있습니다
from pipeline_steps.ingest import process_와_ingest

First run feature store ingestion locally:

In [ ]:
process_와_ingest(input_s3_url, dataset_feature_group.describe()['FeatureGroupArn'])

Define an ingestion pipeline:

In [ ]:
# 피처 스토어 수집 파이프라인을 위한 파라미터 생성
input_s3_url_param = ParameterString(
    name="InputDataUrl",
    default_value=input_s3_url,
)

feature_group_name_param = ParameterString(
    name="FeatureGroupName",
    default_value=dataset_feature_group.describe()['FeatureGroupArn'],
)

In [ ]:
# 피처 스토어 수집 단계
fs_ingest = step(
    process_와_ingest, 
    name=f'{project}-fs-ingest',
)(
    input_s3_url=input_s3_url_param,
    feature_group_name=feature_group_name_param,
)

# 수집 단계가 있는 파이프라인 생성
pipeline_fs_ingest = Pipeline(
    name=f"{pipeline_name}-fs-ingest",
    parameters=[
        input_s3_url_param,
        feature_group_name_param
    ],
    steps=[fs_ingest]
)

In [ ]:
pipeline_fs_ingest.upsert(role_arn=sm_role)

이 피처 수집 파이프라인은 한 단계로만 구성됩니다. 다음 셀에서 생성된 링크를 따라가면 Studio UX에서 파이프라인을 볼 수 있습니다.

In [ ]:
# 파이프라인 링크 표시
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/graph">the pipeline</a> 폴더의 Studio UI</b>'.format(
            domain_id, region, pipeline_fs_ingest.name))
)

In [ ]:
execution_fs_ingest = pipeline_fs_ingest.start()

In [ ]:
execution_fs_ingest.describe()

<div class="alert alert-info">You need to wait until the pipeline completes 와 ingested features into the feature group. 실행은 약 5분이 소요됩니다.</div>

In [ ]:
# 피처 수집 파이프라인 실행이 완료될 때까지 기다려야 합니다
execution_fs_ingest.wait()
execution_fs_ingest.list_steps()

In [ ]:
assert execution_fs_ingest.list_steps()[0]['StepStatus'] == 'Succeeded', 'Ingestion pipepline execution status must be Succeeded!'

### Studio UI의 Feature Store
You can explore feature store 와 feature groups 폴더의 Studio UI. Navigate to **Data** > **Feature Store**:

![](img/feature-store-studio-ui.png)

In the UI you can explore features, feature group metadata, see sample queries, 와 associated pipeline executions.

### 피처 그룹에서 수집된 피처 검색

There are many approaches to extract features from the offline feature store. For example, you can use [Amazon Athena query](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-create-a-dataset.html#feature-store-athena-sample-queries) to query 와 join data stored 폴더의 offline store, or you can use [Offline Store Python SDK](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-create-a-dataset.html#feature-store-dataset-python-sdk). You're going to use Python SDK to extract features 와 create a dataset for the model building pipeline.

<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <p style=" text-align: center; margin: auto;">Ingestion to the offline store is buffered 와 it takes up to 15 minutes for data to appear 폴더의 feature group. After features are ingested 와 available 폴더의 offline store, you can query them 와 create datasets for model training 와 scoring.
    </p>
</div>


In [ ]:
sagemaker_client = boto3.client('sagemaker')
output_location = f's3://{bucket_name}/{feature_store_bucket_prefix}/offline-store/query_results/'

In [ ]:
def get_historical_record_count(fg):
    fs_query = dataset_feature_group.athena_query()
    query_string = f'SELECT COUNT(*) FROM "' + fs_query.table_name + f'"'
    output_location =  f's3://{bucket_name}/{feature_store_bucket_prefix}/offline-store/query_results/'

    fs_query.run(query_string=query_string, output_location=output_location)
    fs_query.wait()
    fs_df = fs_query.as_dataframe()
    
    return fs_df.iat[0, 0]

<div class="alert alert-info">The next code cell waits until features appeared 폴더의 offline store. 최대 15분이 소요될 수 있습니다. If you have already ingested features before, the cell will exit after the first query.</div>

In [ ]:
# 피처 데이터에 액세스하기 전에 오프라인 피처 스토어가 채워졌는지 확인해야 합니다
offline_store_contents = None
while offline_store_contents is None:    
    fs_record_count = get_historical_record_count(dataset_feature_group)
    print(f"Total number of historical record 폴더의 {dataset_feature_group.name}: {fs_record_count}")

    if fs_record_count >= record_count:
        print(f'[{fs_record_count} feature records are available in offline store for {dataset_feature_group.name} feature group]')
        offline_store_contents = fs_record_count
    else:
        print('[Waiting for data arrives in offline store ...]')
        time.sleep(60)

#### Use the Amazon SageMaker Python SDK (DatasetBuilder) to query the feature store
이 섹션에서는 다음을 사용하는 방법을 보여줍니다: [`DatasetBuilder`](https://sagemaker.readthedocs.io/en/stable/api/prep_data/feature_store.html#sagemaker.feature_store.dataset_builder.DatasetBuilder) 를 사용하여 피처 그룹에서 데이터를 가져옵니다. Refer to the [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-create-a-dataset.html) for detailed examples.

In [ ]:
from sagemaker.feature_store.feature_store import FeatureStore

In [ ]:
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

s3_client = boto3.client('s3', region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(service_name="sagemaker-featurestore-runtime",region_name=region)

In [ ]:
# FeatureStore 세션 객체 생성
feature_store_session = sagemaker.Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

feature_store = FeatureStore(sagemaker_session=feature_store_session)

In [ ]:
included_feature_names = [f.feature_name for f in dataset_feature_group.feature_definitions]

In [ ]:
# 각 레코드의 최신 버전을 검색할 데이터셋 빌더 생성
builder = feature_store.create_dataset(
    base=dataset_feature_group,
    # included_feature_names=included_feature_names,
    output_path=output_location,
).처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다._number_of_recent_records_by_record_identifier(1)

In [ ]:
df_dataset, query = builder.to_dataframe()

In [ ]:
df_dataset

### 모델 빌드 파이프라인에 피처 그룹 통합
So far you ingested all transformed features into the feature store. As the last step in this 노트북 you need to adapt the model building pipeline to use the transformed features from the feature group instead of loading 와 transforming a raw data file from an S3 bucket.

The code for feature exraction from the feature store 와 preparing datasets for training, test, validation, 와 quality baseline is 폴더의 Python file `extract.py` 폴더의 folder `./pipeline_steps`. 
Note, there is no feature processing code 폴더의 script because all feature engineering is done before ingesting features into feature store. 

In [ ]:
# Python 함수 코드는 로컬 파일에 있습니다
from pipeline_steps.extract import prepare_datasets

In [ ]:
# 모든 것이 작동하는지 확인하기 위해 함수를 로컬에서 실행
r_extract = prepare_datasets(
    feature_group_name=dataset_feature_group_name,
    output_s3_prefix=output_s3_prefix,
    query_output_s3_path=output_location,
    tracking_server_arn=mlflow_arn,
    experiment_name=f"local-test-{current_timestamp}"
)
r_extract

방금 이 스크립트를 로컬에서 테스트했으므로 이제 모델 빌드 파이프라인에 통합해 보겠습니다. The next cell contains the full script for pipeline building.

For clarity, the batch monitoring 와 quality monitoring steps are omitted from this pipeline. 다음 섹션의 코드를 사용하여 이러한 단계를 자유롭게 추가하세요: **추가 a batch transform 와 quality checks to the pipeline**.

In [ ]:
from pipeline_steps.evaluate import evaluate
from pipeline_steps.register import register

In [ ]:
session = PipelineSession()
experiment_name = f"{project}-fs-pipeline-{current_timestamp}"
mlflow.set_experiment(experiment_name)

# 피처 스토어에서 피처 추출 단계
fs_step_extract_featureset = step(
    prepare_datasets, 
    instance_type=process_instance_type_param,
    name=f"{project}-extract-featureset",
)(
    feature_group_name=feature_group_name_param,
    output_s3_prefix=output_s3_prefix,
    query_output_s3_path=output_location,
    tracking_server_arn=tracking_server_arn_param,
    experiment_name=experiment_name,
    pipeline_run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
)

cache_config = CacheConfig(enable_caching=True)
cache_config.expire_after = "p30d"

# 훈련 단계
fs_step_train = TrainingStep(
    name=f"{project}-train",
    step_args=get_xgb_estimator(
        session=session,
        instance_type=train_instance_type_param,
        output_s3_url=output_s3_url,
        base_job_name=f"{project}-train",
    ).fit(
        {
            "train": TrainingInput(
                fs_step_extract_featureset['train_data'],
                content_type="text/csv",
            ),
            "validation": TrainingInput(
                fs_step_extract_featureset['validation_data'],
                content_type="text/csv",
            ),
        }
    ),
    cache_config=cache_config,
)    

# 평가 단계
fs_step_evaluate = step(
    evaluate,
    instance_type=process_instance_type_param,
    name=f"{project}-evaluate",
)(
    test_x_data_s3_path=fs_step_extract_featureset['test_x_data'],
    test_y_data_s3_path=fs_step_extract_featureset['test_y_data'],
    model_s3_path=fs_step_train.properties.ModelArtifacts.S3ModelArtifacts,
    output_s3_prefix=output_s3_prefix,
    tracking_server_arn=tracking_server_arn_param,
    experiment_name=fs_step_extract_featureset['experiment_name'],
    pipeline_run_id=fs_step_extract_featureset['pipeline_run_id'],
)

# 모델 등록 단계
fs_step_register = step(
        register,
        name=f"{project}-register",
    )(
        training_job_name=fs_step_train.properties.TrainingJobName,
        model_package_group_name=model_package_group_name_param,
        model_approval_status=model_approval_status_param,
        evaluation_result=fs_step_evaluate['evaluation_result'],
        output_s3_prefix=output_s3_url,
        tracking_server_arn=tracking_server_arn_param,
        experiment_name=fs_step_extract_featureset['experiment_name'],
        pipeline_run_id=fs_step_extract_featureset['pipeline_run_id'],
    )

# 파이프라인 실행 실패 단계
fs_step_fail = FailStep(
    name=f"{project}-fail",
    error_message=Join(on=" ", values=["Execution failed due to AUC Score < ", test_score_threshold_param]),
)

# 조건 단계에서 확인할 조건
fs_condition_gte = ConditionGreaterThanOrEqualTo(
        left=fs_step_evaluate['evaluation_result']['classification_metrics']['auc_score']['value'],  
        right=test_score_threshold_param,
)

# 조건부 등록 단계
fs_step_conditional_register = ConditionStep(
    name=f"{project}-check-metrics",
    conditions=[fs_condition_gte],
    if_steps=[fs_step_register],
    else_steps=[fs_step_fail],
)

# 파이프라인 객체 생성
pipeline_feature_store = Pipeline(
    name=f"{pipeline_name}-fs",
    parameters=[
        feature_group_name_param,
        process_instance_type_param,
        train_instance_type_param,
        model_approval_status_param,
        test_score_threshold_param,
        model_package_group_name_param,
        tracking_server_arn_param,
    ],
    steps=[fs_step_conditional_register],
    pipeline_definition_config=PipelineDefinitionConfig(use_custom_job_prefix=True)
)

In [ ]:
pipeline_feature_store.upsert(role_arn=sm_role)

새 파이프라인은 첫 번째 파이프라인과 정확히 동일하게 보이지만 피처 엔지니어링 대신 피처셋 추출 단계가 있습니다 - `from-idea-to-prod-extract-featureset`. To see the pipeline 폴더의 Studio UX click on the link constructed by the following cell.

In [ ]:
# 파이프라인 링크 표시
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/pipelines/{}/graph">the pipeline</a> 폴더의 Studio UI</b>'.format(
            domain_id, region, pipeline_feature_store.name))
)

### 새 파이프라인 실행

In [ ]:
execution_feature_store = pipeline_feature_store.start()

In [ ]:
# 실행이 완료될 때까지 기다리려면 이 두 줄의 주석 해제
# execution_feature_store.wait()
# execution_feature_store.list_steps()

## 추가 읽기: 피처 변환 및 수집을 위한 Feature Processor 사용
SageMaker provides you 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. a Spark-based [Feature Processor SDK](https://sagemaker.readthedocs.io/en/stable/api/prep_data/feature_store.html#feature-processor-decorator) 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. which you can transform 와 ingest data from batch data sources into your feature groups. Read through the description 와 examples 폴더의 [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-feature-processing.html).

Refer to a more detailed example of feature processor in [feature store feature processor](https://github.com/aws/amazon-sagemaker-examples/blob/main/sagemaker-featurestore/feature_store_feature_processor.ipynb) 노트북.

## 요약
In this 노트북 you've built three SageMaker pipelines:
- An initial model building pipeline 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. data processing, model training, model evaluation, 와 conditional model registration steps
- Next version of this pipeline 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. data 와 model quality monitoring 와 batch transform
- A feature engineering 와 ingestion into the SageMaker Feature Store pipeline
- An adapted version of the initial pipeline 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the data processing step replaced by the featureset extraction step.

---

## 워크샵 진행 계속
After finishing this lab, you can continue 처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다. the step 4 와 5 [노트북s](04-sagemaker-project.ipynb) or go directly to the step 6 [노트북](06-monitoring.ipynb):

- **Step 4 와 5 노트북s**: Use SageMaker Projects to implement CI/CD automation pipelines for model build 와 deployment

- **Step 6 노트북s**: Data 와 model quality monitoring

## 실제 프로젝트를 위한 추가 개발 아이디어
- 추가 [bias detection 와 model explainability](https://docs.aws.amazon.com/sagemaker/latest/dg/build-와-manage-steps.html#step-type-clarify-check) steps. 추가 model metrics calculated by [SageMaker Clarify](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-configure-processing-jobs.html) to the model metadata 폴더의 model registry
- 추가 event-driven launching of the ML pipeline as soon as a new dataset is uploaded to an Amazon S3 bucket. 다음을 사용할 수 있습니다: [Amazon EventBridge integeration](https://docs.aws.amazon.com/sagemaker/latest/dg/pipeline-eventbridge.html#pipeline-eventbridge-schedule) to implement various event-driven workflows
- Use a designated IAM execution role for the pipeline execution
- 추가 data encryption by using S3 bucket encryption 와 AWS KMS keys for container EBS volume encryption

## 추가 리소스
- [Automate Machine Learning Workflows](https://aws.amazon.com/getting-started/h와s-on/machine-learning-tutorial-mlops-automate-ml-workflows/)
- [Amazon SageMaker Feature Store workshop](https://github.com/aws-samples/amazon-sagemaker-feature-store-end-to-end-workshop)
- [Amazon SageMaker Model Building Pipeline](https://github.com/aws/sagemaker-python-sdk/blob/master/doc/amazon_sagemaker_model_building_pipeline.rst)
- [MLOPs With SageMaker Pipelines Step Decorator](https://towardsaws.com/mlops-처럼 즉시가 아니라 파이프라인 실행 시 작업을 시작해야 합니다.-sagemaker-pipelines-step-decorator-bb63fce88846)

# 커널 종료

In [ ]:
%%html

<p><b>Shutting down your kernel for this 노트북 to release resources.</b></p>
<button class="sm-comm와-button" data-comm와linker-comm와="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-comm와-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>